# 02 — Raw data validation

Validates the real raw CSVs before anything is derived from them: universe size, coverage by decade, per-field missingness, and — most importantly — the point-in-time integrity of `available_date`, the timestamp the whole no-lookahead guarantee rests on.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## Reading the real fundamentals export

`read_fundamentals_quarterly` parses the wide CSV into `RawFundamentalsRow` records. A blank cell becomes `None` — a real, disclosed missing value, never a fabricated `0.0`.

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.acquisition.fundamentals_quarterly import (
        read_fundamentals_quarterly,
    )

    raw_rows = read_fundamentals_quarterly(rd.RAW_ROOT / "fundamentals_quarterly.csv")
    quarter_ends = sorted({row.quarter_end for row in raw_rows})
    print("rows            ", f"{len(raw_rows):,}")
    print("distinct tickers", f"{len({row.symbol for row in raw_rows}):,}")
    print("period_end range", quarter_ends[0], "..", quarter_ends[-1])
    print("distinct quarters", len(quarter_ends))
else:
    raw_rows = ()
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Coverage by decade

The universe is not constant through history: it grows from a handful of names in the 1980s to the full 1,520 today. Any statistic computed over the early history is dominated by that, which is why the IC backtest excludes thin cross-sections from its headline stats.

In [ ]:
if AVAILABLE:
    from collections import Counter

    rows_by_decade = Counter(row.quarter_end.year // 10 * 10 for row in raw_rows)
    tickers_by_decade = {}
    for row in raw_rows:
        tickers_by_decade.setdefault(row.quarter_end.year // 10 * 10, set()).add(row.symbol)

    print(f"{'decade':>8} {'rows':>10} {'tickers':>9}")
    for decade in sorted(rows_by_decade):
        print(f"{decade:>8} {rows_by_decade[decade]:>10,} {len(tickers_by_decade[decade]):>9,}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Point-in-time integrity of `available_date`

`filed_at` carries each row's `available_date`. Two properties must hold, or the whole no-lookahead guarantee is void:

1. `filed_at >= quarter_end` for every row — a quarter's data cannot be knowable before the quarter it reports on has ended. `FundamentalsFeatureRecord.__post_init__` enforces this and rejects violators at normalization rather than letting them through.
2. The reporting lag must be plausible, not zero.

In [ ]:
if AVAILABLE:
    import statistics

    lags = [(row.filed_at.date() - row.quarter_end).days for row in raw_rows]
    violations = [lag for lag in lags if lag < 0]
    estimated = sum(1 for row in raw_rows if row.filed_at_is_estimated)

    print("rows with filed_at before quarter_end:", len(violations))
    print()
    print("reporting lag (days), filed_at - quarter_end:")
    print("  min   ", min(lags))
    print("  median", statistics.median(lags))
    print("  mean  ", round(statistics.mean(lags), 1))
    print("  max   ", max(lags))
    print()
    print(f"rows with an ESTIMATED available_date: {estimated:,} of {len(raw_rows):,} "
          f"({estimated / len(raw_rows):.1%})")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Structural validation of the normalized records

`normalize_fundamentals_batch` drops any row a Stage 3 domain type rejects and reports it as a `DataValidationIssue` — it never crashes the run and never substitutes a default in place of a rejected fact. `validate_filings` then checks the normalized records.

In [ ]:
if AVAILABLE:
    from collections import Counter

    from atlas_quant.strategies.multi_factor_ranking_ml.production.normalization import (
        normalize_fundamentals_batch,
    )
    from atlas_quant.strategies.multi_factor_ranking_ml.production.validation import validate_filings

    records, normalization_issues = normalize_fundamentals_batch(raw_rows)
    filing_issues = validate_filings(records)

    print(f"normalized {len(records):,} of {len(raw_rows):,} raw rows")
    print(f"rejected during normalization: {len(normalization_issues):,}")
    print(f"validate_filings issues:       {len(filing_issues):,}")
    print()
    for label, issues in (("normalization", normalization_issues), ("validate_filings", filing_issues)):
        by_kind = Counter((issue.severity.value, issue.category) for issue in issues)
        for (severity, category), count in by_kind.most_common(10):
            print(f"  {label:<16} {severity:<8} {category:<16} {count:>6,}")
    if filing_issues:
        print()
        print("example:", filing_issues[0].subject, "|", filing_issues[0].message[:120])
else:
    records = ()
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Per-feature missingness across the real universe

Missing values are genuinely missing — they are carried as NaN into the model matrix, never imputed (see notebook 04).

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.feature_domain import FEATURE_NAMES

    recent = [r for r in records if r.quarter_end.year >= 2015]
    present = {name: 0 for name in FEATURE_NAMES}
    for record in recent:
        for name in FEATURE_NAMES:
            if record.features.get(name) is not None:
                present[name] += 1

    coverage = sorted((count / len(recent), name) for name, count in present.items())
    print(f"coverage over {len(recent):,} rows with quarter_end >= 2015")
    print()
    print("thinnest 10 features:")
    for rate, name in coverage[:10]:
        print(f"  {name:<40} {rate:>7.1%}")
    print()
    print("fully-covered features:", sum(1 for rate, _ in coverage if rate >= 0.999))
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## The daily price export

Scanned directly (streaming, without materializing 9.2M records) to confirm shape and density. A blank cell means no trade/no data that day and is omitted entirely by the parser — never coerced to `0.0` or forward-filled.

In [ ]:
if AVAILABLE:
    import csv

    price_path = rd.RAW_ROOT / "Close_Price.csv"
    with price_path.open(newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        ticker_columns = len(header) - 1
        row_count = 0
        filled_cells = 0
        first_date = last_date = None
        for row in reader:
            row_count += 1
            if first_date is None:
                first_date = row[0]
            last_date = row[0]
            filled_cells += sum(1 for cell in row[1:] if cell.strip())

    print("ticker columns", f"{ticker_columns:,}")
    print("trading dates ", f"{row_count:,}", f"({first_date} .. {last_date})")
    print("populated cells", f"{filled_cells:,} of {row_count * ticker_columns:,}",
          f"({filled_cells / (row_count * ticker_columns):.1%})")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Findings

- The fundamentals export covers 1,520 tickers over ~46 years of quarters, with coverage growing by decade — the modern cross-section is roughly 1,400 usable names per quarter, the 1980s cross-section only a handful.
- No row has `filed_at` before its own `quarter_end`: the point-in-time ordering the pipeline depends on holds across the whole real dataset.
- Missingness is real and unevenly distributed across features. It is reported, not filled.

## Limitations

- `available_date` is estimated for part of the history rather than observed. Where it is estimated, the point-in-time cutoff is only as good as that estimate — classified `data_provenance_required` in `docs/reproducibility_findings.md`.
- The universe is survivorship-biased: it is the set of tickers present in the export as pulled, not the set investable at each historical date. Every historical statistic in this series inherits that bias.
- This notebook validates structure and timing. It cannot validate that Bloomberg's underlying values are correct.